In [37]:
import numpy as np
import pandas as pd

# Loading Data

In [38]:
df_train_origin = pd.read_csv('data/train.csv') # With Target
df_test_origin= pd.read_csv('data/test.csv') # No Target
df_train_origin.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [39]:
df_train_origin.dtypes

PassengerId         str
HomePlanet          str
CryoSleep        object
Cabin               str
Destination         str
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name                str
Transported        bool
dtype: object

In [40]:
x = df_test_origin['Age']

# Removing Irrelevant Data

In [ ]:
SPENDING_COLS = [
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck'
]

DROP_COLS = [
    'PassengerId',
    'Name'
]


def preprocess_data(df, medians=None, train_columns=None):
    df = df.copy()

    # Remove irrelevant columns
    df = df.drop(columns=DROP_COLS)

    # Feature engineering: Cabin
    df['CabinDeck'] = df['Cabin'].str.split('/').str[0]
    df['CabinSide'] = df['Cabin'].str.split('/').str[2]
    df = df.drop(columns='Cabin')

    # Feature engineering: Total spending
    df['TotalSpendings'] = df[SPENDING_COLS].sum(axis=1)

    # Numerical missing values
    numeric_cols = SPENDING_COLS + ['Age']

    if medians is None:
        medians = df[numeric_cols].median()

    df[numeric_cols] = df[numeric_cols].fillna(medians)

    # Categorical missing values
    categorical_cols = df.select_dtypes(
        include=['object', 'string']
    ).columns

    df[categorical_cols] = df[categorical_cols].fillna('Unknown')

    # One-hot encode categorical features
    categorical_cols = df.select_dtypes(
        include=['object', 'string']
    ).columns

    df = pd.get_dummies(
        df,
        columns=categorical_cols,
        drop_first=True,
        dtype=int
    )

    # Make test/validation columns match training columns
    if train_columns is not None:
        df = df.reindex(columns=train_columns, fill_value=0)

    return df, medians

X_train = df_train_origin.drop(columns='Transported')
y_train = df_train_origin['Transported']

X_train, median = preprocess_data(X_train)
X_test, _ = preprocess_data(
    df_test_origin,
    median,
    X_train.columns
)

print(X_train.shape, X_test.shape)
X_test.head()


RoomService      0.0
FoodCourt        0.0
ShoppingMall     0.0
Spa              0.0
VRDeck           0.0
Age             27.0
dtype: float64
RoomService      0.0
FoodCourt        0.0
ShoppingMall     0.0
Spa              0.0
VRDeck           0.0
Age             27.0
dtype: float64
(8693, 27) (4277, 27)


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,TotalSpendings,HomePlanet_Europa,HomePlanet_Mars,HomePlanet_Unknown,...,CabinDeck_B,CabinDeck_C,CabinDeck_D,CabinDeck_E,CabinDeck_F,CabinDeck_G,CabinDeck_T,CabinDeck_Unknown,CabinSide_S,CabinSide_Unknown
0,27.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,...,0,0,0,0,0,1,0,0,1,0
1,19.0,0.0,9.0,0.0,2823.0,0.0,2832.0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
2,31.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0,...,0,1,0,0,0,0,0,0,1,0
3,38.0,0.0,6652.0,0.0,181.0,585.0,7418.0,1,0,0,...,0,1,0,0,0,0,0,0,1,0
4,20.0,10.0,0.0,635.0,0.0,0.0,645.0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
